# database_comparisons.ipynb
What pathogenic variants are present in each database?  
Variant can be uniquely identified by Gene, AA Position, AA Reference, AA Alternate  
How many variants are shared?  
How many variants are unique in a data set?  
Are there any patterns?  

RESULT  
1386 variants are shared.  
46625 variants are only in CMC.  
2113 variants are only in hotspots.  
Shared variants are mostly missense or nonsense mutation.  
However, we couldn't deal with all variants properly because of their different annotation.  

Other notes:  
- CMC too big for Excel  

## import and format

In [ ]:
import pandas as pd
pd.set_option('display.max_columns', None)
import re


In [ ]:
def load_hotspots_SNV_table(file ='../../data/variants/databases/hotspots_v2.xls'):
    df = pd.read_excel(file, sheet_name=0)
    
    return df   
def parse_hotspots_SNV_table(df):
    df = df.loc[:, ['Hugo_Symbol','Amino_Acid_Position','Reference_Amino_Acid','Variant_Amino_Acid']]
    df['Reference_Amino_Acid'] = df['Reference_Amino_Acid'].str.split(':').str[0]
    df['Variant_Amino_Acid'] = df['Variant_Amino_Acid'].str.split(':').str[0]
    df['var_id'] = df['Hugo_Symbol'] + '/' + df['Reference_Amino_Acid'] + df['Amino_Acid_Position'] + df['Variant_Amino_Acid']
    return df
    
def load_hotspots_INDEL_table(file ='../../data/variants/databases/hotspots_v2.xls'):
    df = pd.read_excel(file, sheet_name=1)
    return df
def parse_hotspots_INDEL_table(df):
    df = df.loc[:,['Hugo_Symbol','Variant_Amino_Acid']]
    df['Variant_Amino_Acid'] = df['Variant_Amino_Acid'].str.split(':').str[0]
    df['var_id'] = df['Hugo_Symbol'] + '/' + df['Variant_Amino_Acid']
    return df

def parse_hotspots_table(SNV,INDEL):
    df = pd.concat([parse_hotspots_SNV_table(SNV),parse_hotspots_INDEL_table(INDEL)], ignore_index=True)
    df['var_id'] = df['var_id'].str.replace(' ','')
    return df

def load_pathogenic_cmc_table(file ='../../data/variants/databases/CancerMutationCensus_AllData_v102_GRCh37.tsv'):
    df = pd.read_table(file)
    df = df[df['MUTATION_SIGNIFICANCE_TIER'].isin(['1','2','3'])]
    return df

def parse_cmc_table(df):
    df = df.loc[:, ['GENE_NAME','Mutation AA','Mutation Description AA']]
    df['Mutation AA'] = df['Mutation AA'].str.split('.').str[1]
    return df
# define some column var_id including all information necessary to uniquely identify a variant:
# gene, AA position, AA ref, AA alt


In [ ]:
cmc = load_pathogenic_cmc_table()
df_cmc = parse_cmc_table(cmc)
hs_SNV = load_hotspots_SNV_table()
hs_INDEL = load_hotspots_INDEL_table()

In [ ]:
def normalize_cmc(row):
    gene = str(row["GENE_NAME"]).upper().replace(" ", "")
    MA = str(row["Mutation AA"]).replace(" ", "")
    vtype = str(row.get("Mutation Description AA", "")).lower()
    # サイレント変異 "=" を ref に置換
    if MA.endswith("="):
        MA = MA[:-1] + MA[0]

    # frameshift deletion / insertion / delins を簡略
    if 'fs*' in MA.lower():
        m = re.match(r"([A-Z])(\d+)", MA)
        if m:       
            if vtype == "deletion - frameshift":
                return f"{gene}/{m.group(1)}{m.group(2)}del"
            elif vtype == "insertion - frameshift":
                return f"{gene}/{m.group(1)}{m.group(2)}ins"
            elif vtype == "complex - frameshift":
                return f"{gene}/{m.group(1)}{m.group(2)}delins"
                
    return f"{gene}/{MA}"
df_cmc['var_id'] = df_cmc.apply(normalize_cmc, axis=1)


## count
count how many variants are shared, only in CMC, only in hotspots  
detect more common variant type  

In [ ]:
a = df_cmc
b = parse_hotspots_table(hs_SNV,hs_INDEL)

# Intersect is given by:
both_dict = set(a['var_id']) & set(b['var_id'])

# Variants in cmc and not in hotspots:
cmc_only_dict = set(a['var_id']) - set(b['var_id'])

# Variants in hotspots and not in cmc:
hot_only_dict = set(b['var_id']) - set(a['var_id'])

In [ ]:
print(len(set(a['var_id']) & set(b['var_id'])),'variants are shared')
print(len(set(a['var_id']) - set(b['var_id'])),'variants are only in cmc')
print(len(set(b['var_id']) - set(a['var_id'])),'variants are only in hotspots')


In [ ]:
result = {
    'both': both_dict,
    'cmc_only': cmc_only_dict,
    'hot_only': hot_only_dict
}

data = []
for category, var_set in result.items():
    for var in var_set:
        data.append({'category':category, 'var_id':var})

df = pd.DataFrame(data)

In [ ]:
def classify_variant(var_id):
    v = var_id.lower()
    m = re.match(r'([A-Za-z0-9-_.]+)/([A-Z*])(\d+)([A-Z*])',var_id)
    
    if 'splice' in v or 'ivs' in v:
        return 'splice'
    if 'dup'in v:
        return 'duplication'
    if 'delins'in v:
        return 'delins'
    if 'del' in v:
        return 'deletion'
    if 'ins' in v:
        return 'insertion'
    m = re.match(r'([A-Za-z0-9-_.]+)/([A-Z*])(\d+)([A-Z*])',var_id)    
    if m:
        ref,pos,alt =m.group(2),m.group(3),m.group(4)
        if ref == alt:
            return 'silent'
        if alt == '*':
            return 'nonsense'
        elif ref != alt:
            return 'missense'
    return 'other'

In [ ]:
df['variant_type'] = df['var_id'].apply(classify_variant)

In [ ]:
table = df.groupby(['variant_type','category']).size().unstack(fill_value=0)
print(table)

In [ ]:
df['Gene'] = df['var_id'].str.split('/').str[0]

hot_genes = df[df['category']=='hot_only']['Gene'].value_counts()
cmc_genes = df[df['category']=='cmc_only']['Gene'].value_counts()
both_genes = df[df['category']=='both']['Gene'].value_counts()

In [ ]:
print('Hotspots top',hot_genes.head(10))
print('CMC top',cmc_genes.head(10))
print('Both top',both_genes.head(10))